In [1]:
import numpy as np
import xarray as xr
from scipy.signal import detrend
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from sst_pix2pix_annual_cycle import (
    SSTDataset, Generator, compute_climatology, 
    rollout_prediction, get_seasonal_encoding
)
from train_sst_model import (
    Discriminator, CombinedLoss, 
    plot_training_history, plot_predictions
)

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# ============================================================================
# CMIP6 Data Processing
# ============================================================================

def load_and_process_cmip6_data(dataset_path, mask_path='../../Data/mask.npy', 
                                 n_train=1428, n_val=100, ensemble_id=0):
    """
    Load and process CMIP6 SST data.
    
    Args:
        dataset_path: Path to CMIP6 netCDF file
        mask_path: Path to mask file
        n_train: Number of training samples
        n_val: Number of validation samples
        ensemble_id: Which ensemble member to use (if multiple)
    
    Returns:
        train_data, val_data, test_data, train_months, val_months, test_months, mask
    """
    print(f"Loading dataset from: {dataset_path}")
    
    # Load dataset
    ds = xr.open_dataset(dataset_path)
    
    # Select SST variable
    sst = ds["sst"]
    
    # If multiple ensembles, select one
    if "ensemble" in sst.dims:
        print(f"Multiple ensembles found. Selecting ensemble {ensemble_id}")
        sst = sst.isel(ensemble=ensemble_id)
    
    # Fill NaN values with 0
    sst = sst.fillna(0)
    
    # Stack years and months into time dimension
    sst_time = (
        sst
        .stack(time=("years", "mon"))
        .transpose("time", "lat", "lon")
    ).fillna(0)
    
    data = sst_time.values  # Shape: (time, lat, lon)
    
    print(f"Data shape: {data.shape}")
    print(f"Data range: [{data.min():.2f}, {data.max():.2f}]")
    
    # Load mask
    mask = np.load(mask_path)
    print(f"Mask shape: {mask.shape}")
    
    # Split data
    train_data = data[:n_train, :, :]
    val_data = data[n_train:n_train + n_val, :, :]
    test_data = data[n_train:, :, :]
    
    print(f"Train shape: {train_data.shape}")
    print(f"Val shape: {val_data.shape}")
    print(f"Test shape: {test_data.shape}")
    
    # Create month arrays (0-11 for each sample)
    # Assuming data starts from a known month, or extract from dataset
    # If your dataset has months encoded, extract them. Otherwise:
    total_samples = data.shape[0]
    
    # Try to get starting month from dataset metadata
    if 'mon' in ds.coords:
        # If months are in coordinates, use them
        months_all = np.array([(i % 12) for i in range(total_samples)])
    else:
        # Default: assume data starts in January (month 0)
        months_all = np.array([i % 12 for i in range(total_samples)])
    
    train_months = months_all[:n_train]
    val_months = months_all[n_train:n_train + n_val]
    test_months = months_all[n_train:]
    
    return train_data, val_data, test_data, train_months, val_months, test_months, mask


def process_with_detrending(data, detrend_type='linear'):
    """
    Optional: Apply detrending to SST data.
    
    Args:
        data: SST data (time, lat, lon)
        detrend_type: 'linear' or 'constant'
    
    Returns:
        detrended data
    """
    if detrend_type is None:
        return data
    
    print(f"Applying {detrend_type} detrending...")
    data_detrended = np.zeros_like(data)
    
    # Detrend along time axis for each spatial point
    # for i in range(data.shape[1]):
    #     for j in range(data.shape[2]):
    #         data_detrended[:, i, j] = detrend(data[:, i, j], type=detrend_type)

    data_detrended = detrend(data, axis=0, type=detrend_type)
    
    return data_detrended


def normalize_data(train_data, val_data, test_data, method='standardize'):
    """
    Normalize SST data.
    
    Args:
        train_data, val_data, test_data: SST arrays
        method: 'standardize' (z-score) or 'minmax' (0-1 scaling)
    
    Returns:
        normalized data and normalization parameters
    """
    if method == 'standardize':
        mean = np.mean(train_data)
        std = np.std(train_data)
        
        train_norm = (train_data - mean) / std
        val_norm = (val_data - mean) / std
        test_norm = (test_data - mean) / std
        
        norm_params = {'mean': mean, 'std': std, 'method': 'standardize'}
        
        print(f"Standardization - Mean: {mean:.4f}, Std: {std:.4f}")
        
    elif method == 'minmax':
        min_val = np.min(train_data)
        max_val = np.max(train_data)
        
        train_norm = (train_data - min_val) / (max_val - min_val)
        val_norm = (val_data - min_val) / (max_val - min_val)
        test_norm = (test_data - min_val) / (max_val - min_val)
        
        norm_params = {'min': min_val, 'max': max_val, 'method': 'minmax'}
        
        print(f"Min-Max scaling - Min: {min_val:.4f}, Max: {max_val:.4f}")
    
    else:
        # No normalization
        train_norm, val_norm, test_norm = train_data, val_data, test_data
        norm_params = {'method': 'none'}
    
    return train_norm, val_norm, test_norm, norm_params


def denormalize_data(data, norm_params):
    """
    Denormalize data back to original scale.
    
    Args:
        data: normalized data
        norm_params: dictionary with normalization parameters
    
    Returns:
        denormalized data
    """
    if norm_params['method'] == 'standardize':
        return data * norm_params['std'] + norm_params['mean']
    elif norm_params['method'] == 'minmax':
        return data * (norm_params['max'] - norm_params['min']) + norm_params['min']
    else:
        return data


In [3]:
# ============================================================================
# Training with CMIP6 Data
# ============================================================================

def train_cmip6_model(dataset_path, mask_path='../Data/mask.npy',
                      n_train=1428, n_val=100, ensemble_id=0,
                      num_epochs=100, num_input_months=3, 
                      batch_size=16, lr=0.0002, lambda_clim=0.1,
                      normalize=True, apply_detrend=None,
                      checkpoint_dir='./checkpoints'):
    """
    Complete training pipeline for CMIP6 SST data.
    
    Args:
        dataset_path: Path to CMIP6 netCDF file
        mask_path: Path to mask file
        n_train: Number of training samples
        n_val: Number of validation samples
        ensemble_id: Which ensemble to use
        num_epochs: Training epochs
        num_input_months: Number of previous months as input
        batch_size: Batch size
        lr: Learning rate
        lambda_clim: Climatology loss weight
        normalize: Whether to normalize data
        apply_detrend: 'linear', 'constant', or None
        checkpoint_dir: Directory to save checkpoints
    """
    import os
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}\n")
    
    # ===========================
    # 1. Load and Process Data
    # ===========================
    train_data, val_data, test_data, train_months, val_months, test_months, mask = \
        load_and_process_cmip6_data(dataset_path, mask_path, n_train, n_val, ensemble_id)
    
    # Optional detrending
    if apply_detrend:
        train_data = process_with_detrending(train_data, apply_detrend)
        val_data = process_with_detrending(val_data, apply_detrend)
        test_data = process_with_detrending(test_data, apply_detrend)
    
    # Normalize data
    if normalize:
        train_data, val_data, test_data, norm_params = normalize_data(
            train_data, val_data, test_data, method='standardize'
        )
    else:
        norm_params = {'method': 'none'}
    
    # ===========================
    # 2. Compute Climatology
    # ===========================
    print("\nComputing climatology from training data...")
    climatology = compute_climatology(train_data, train_months)
    print(f"Climatology shape: {climatology.shape}")
    
    # ===========================
    # 3. Create Datasets
    # ===========================
    print("\nCreating datasets...")
    train_dataset = SSTDataset(
        sst_data=train_data,
        months=train_months,
        num_input_months=num_input_months,
        climatology=climatology
    )
    
    val_dataset = SSTDataset(
        sst_data=val_data,
        months=val_months,
        num_input_months=num_input_months,
        climatology=climatology
    )
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=4,
        pin_memory=True if device == 'cuda' else False
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True if device == 'cuda' else False
    )
    
    print(f"Training samples: {len(train_dataset)}")
    print(f"Validation samples: {len(val_dataset)}")
    
    # ===========================
    # 4. Initialize Models
    # ===========================
    in_channels = num_input_months + 2  # SST channels + seasonal encoding
    print(f"\nInitializing models with {in_channels} input channels...")
    
    generator = Generator(in_channels=in_channels, features=128, out_channels=1).to(device)
    discriminator = Discriminator(in_channels=in_channels + 1).to(device)
    
    # Initialize weights
    def init_weights(m):
        if isinstance(m, torch.nn.Conv2d) or isinstance(m, torch.nn.ConvTranspose2d):
            torch.nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None:
                torch.nn.init.constant_(m.bias, 0)
        elif isinstance(m, torch.nn.BatchNorm2d):
            torch.nn.init.normal_(m.weight, 1.0, 0.02)
            torch.nn.init.constant_(m.bias, 0)
    
    generator.apply(init_weights)
    discriminator.apply(init_weights)
    
    # ===========================
    # 5. Setup Training
    # ===========================
    criterion = CombinedLoss(climatology_data=climatology, lambda_clim=lambda_clim, lambda_l1=100.0)
    bce_loss = torch.nn.BCEWithLogitsLoss()
    
    g_optimizer = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
    
    g_scheduler = torch.optim.lr_scheduler.StepLR(g_optimizer, step_size=30, gamma=0.5)
    d_scheduler = torch.optim.lr_scheduler.StepLR(d_optimizer, step_size=30, gamma=0.5)
    
    # Training history
    history = {
        'train_g_loss': [],
        'train_d_loss': [],
        'train_l1_loss': [],
        'train_clim_loss': [],
        'val_g_loss': [],
        'val_l1_loss': []
    }
    
    # ===========================
    # 6. Training Loop
    # ===========================
    print(f"\nStarting training for {num_epochs} epochs...")
    print("=" * 70)
    
    best_val_loss = float('inf')
    
    for epoch in range(num_epochs):
        # Train
        generator.train()
        discriminator.train()
        
        epoch_g_loss = 0
        epoch_d_loss = 0
        epoch_l1_loss = 0
        epoch_clim_loss = 0
        
        for batch_idx, batch in enumerate(train_loader):
            inputs = batch['input'].to(device)
            targets = batch['target'].to(device)
            target_months = batch['target_month'].to(device)
            
            # Train Discriminator
            d_optimizer.zero_grad()
            
            real_pair = torch.cat([inputs, targets], dim=1)
            d_real = discriminator(real_pair)
            real_labels = torch.ones_like(d_real) * 0.9
            d_real_loss = bce_loss(d_real, real_labels)
            
            fake_outputs = generator(inputs)
            fake_pair = torch.cat([inputs, fake_outputs.detach()], dim=1)
            d_fake = discriminator(fake_pair)
            fake_labels = torch.zeros_like(d_fake) + 0.1
            d_fake_loss = bce_loss(d_fake, fake_labels)
            
            d_loss = (d_real_loss + d_fake_loss) / 2
            d_loss.backward()
            d_optimizer.step()
            
            # Train Generator
            g_optimizer.zero_grad()
            
            fake_outputs = generator(inputs)
            fake_pair = torch.cat([inputs, fake_outputs], dim=1)
            d_fake = discriminator(fake_pair)
            
            real_labels = torch.ones_like(d_fake)
            g_adv_loss = bce_loss(d_fake, real_labels)
            
            g_recon_loss, loss_dict = criterion(fake_outputs, targets, target_months)
            
            g_loss = g_adv_loss + g_recon_loss
            g_loss.backward()
            g_optimizer.step()
            
            epoch_g_loss += g_loss.item()
            epoch_d_loss += d_loss.item()
            epoch_l1_loss += loss_dict['l1']
            epoch_clim_loss += loss_dict['climatology']
            
            if batch_idx % 50 == 0:
                print(f'Epoch {epoch+1}/{num_epochs} [{batch_idx}/{len(train_loader)}] '
                      f'G: {g_loss.item():.4f} D: {d_loss.item():.4f} '
                      f'L1: {loss_dict["l1"]:.4f} Clim: {loss_dict["climatology"]:.4f}')
        
        # Validation
        generator.eval()
        val_g_loss = 0
        val_l1_loss = 0
        
        with torch.no_grad():
            for batch in val_loader:
                inputs = batch['input'].to(device)
                targets = batch['target'].to(device)
                target_months = batch['target_month'].to(device)
                
                fake_outputs = generator(inputs)
                g_recon_loss, loss_dict = criterion(fake_outputs, targets, target_months)
                
                val_g_loss += g_recon_loss.item()
                val_l1_loss += loss_dict['l1']
        
        # Update schedulers
        g_scheduler.step()
        d_scheduler.step()
        
        # Record history
        history['train_g_loss'].append(epoch_g_loss / len(train_loader))
        history['train_d_loss'].append(epoch_d_loss / len(train_loader))
        history['train_l1_loss'].append(epoch_l1_loss / len(train_loader))
        history['train_clim_loss'].append(epoch_clim_loss / len(train_loader))
        history['val_g_loss'].append(val_g_loss / len(val_loader))
        history['val_l1_loss'].append(val_l1_loss / len(val_loader))
        
        avg_val_loss = val_g_loss / len(val_loader)
        
        print(f'\n{"="*70}')
        print(f'Epoch {epoch+1}/{num_epochs} Summary:')
        print(f'Train - G: {history["train_g_loss"][-1]:.4f} D: {history["train_d_loss"][-1]:.4f}')
        print(f'Val   - G: {avg_val_loss:.4f} L1: {history["val_l1_loss"][-1]:.4f}')
        print(f'{"="*70}\n')
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'epoch': epoch,
                'generator_state_dict': generator.state_dict(),
                'discriminator_state_dict': discriminator.state_dict(),
                'g_optimizer_state_dict': g_optimizer.state_dict(),
                'd_optimizer_state_dict': d_optimizer.state_dict(),
                'history': history,
                'climatology': climatology,
                'norm_params': norm_params,
                'num_input_months': num_input_months,
                'best_val_loss': best_val_loss
            }, os.path.join(checkpoint_dir, 'best_model.pth'))
            print(f"✓ Best model saved (val_loss: {best_val_loss:.4f})")
        
        # Save checkpoint every 10 epochs
        if (epoch + 1) % 10 == 0:
            torch.save({
                'epoch': epoch,
                'generator_state_dict': generator.state_dict(),
                'discriminator_state_dict': discriminator.state_dict(),
                'g_optimizer_state_dict': g_optimizer.state_dict(),
                'd_optimizer_state_dict': d_optimizer.state_dict(),
                'history': history,
                'climatology': climatology,
                'norm_params': norm_params,
                'num_input_months': num_input_months
            }, os.path.join(checkpoint_dir, f'checkpoint_epoch_{epoch+1}.pth'))
    
    # Plot training history
    plot_training_history_with_val(history, save_path='training_history.png')
    
    return generator, discriminator, history, climatology, norm_params, test_data, test_months


In [4]:
def plot_training_history_with_val(history, save_path='training_history.png'):
    """Plot training and validation losses."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    axes[0, 0].plot(history['train_g_loss'], label='Train')
    axes[0, 0].plot(history['val_g_loss'], label='Val')
    axes[0, 0].set_title('Generator Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    axes[0, 1].plot(history['train_d_loss'])
    axes[0, 1].set_title('Discriminator Loss (Train)')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].grid(True)
    
    axes[1, 0].plot(history['train_l1_loss'], label='Train')
    axes[1, 0].plot(history['val_l1_loss'], label='Val')
    axes[1, 0].set_title('L1 Loss')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Loss')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    axes[1, 1].plot(history['train_clim_loss'])
    axes[1, 1].set_title('Climatology Loss (Train)')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Loss')
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Training history saved to: {save_path}")
    plt.close()



In [5]:
# ============================================================================
# Main Execution
# ============================================================================

if __name__ == "__main__":
    # Configuration
    # DATASET_PATH = '../../Data/CMIP6-SST/GISS-E2-1-H_historical_1850_2014.nc'
    # Alternative datasets:
    # DATASET_PATH = '../../Data/CMIP6-SST/MIROC6_historical_1850_2014.nc'
    DATASET_PATH = '../../Data/CMIP6-SST/EC_Earth3_CC_historical_1850_2014.nc'
    
    MASK_PATH = '../../Data/mask.npy'
    N_TRAIN = 1428
    N_VAL = 100
    ENSEMBLE_ID = 0
    
    # Training hyperparameters
    NUM_EPOCHS = 200  # Increase for better results
    NUM_INPUT_MONTHS = 3
    BATCH_SIZE = 16
    LR = 0.0002
    LAMBDA_CLIM = 0.1
    
    # Run training
    generator, discriminator, history, climatology, norm_params, test_data, test_months = \
        train_cmip6_model(
            dataset_path=DATASET_PATH,
            mask_path=MASK_PATH,
            n_train=N_TRAIN,
            n_val=N_VAL,
            ensemble_id=ENSEMBLE_ID,
            num_epochs=NUM_EPOCHS,
            num_input_months=NUM_INPUT_MONTHS,
            batch_size=BATCH_SIZE,
            lr=LR,
            lambda_clim=LAMBDA_CLIM,
            normalize=True,
            apply_detrend=None,  # Set to 'linear' if you want detrending
            checkpoint_dir='./checkpoints_EC_Earth3_CC'
        )
    
    print("\nTraining complete!")
    print("Best model saved in: ./checkpoints_EC_Earth3_CC/best_model.pth")


Using device: cuda

Loading dataset from: ../../Data/CMIP6-SST/EC_Earth3_CC_historical_1850_2014.nc
Multiple ensembles found. Selecting ensemble 0
Data shape: (1980, 48, 144)
Data range: [-1.97, 35.21]
Mask shape: (48, 144)
Train shape: (1428, 48, 144)
Val shape: (100, 48, 144)
Test shape: (552, 48, 144)
Standardization - Mean: 9.2070, Std: 11.3309

Computing climatology from training data...
Climatology shape: (12, 48, 144)

Creating datasets...
Training samples: 1425
Validation samples: 97

Initializing models with 5 input channels...


OutOfMemoryError: CUDA out of memory. Tried to allocate 64.00 MiB. GPU 0 has a total capacity of 44.34 GiB of which 14.88 MiB is free. Process 1540842 has 32.63 GiB memory in use. Process 3140901 has 10.22 GiB memory in use. Process 292549 has 1.14 GiB memory in use. Including non-PyTorch memory, this process has 314.00 MiB memory in use. Of the allocated memory 43.20 MiB is allocated by PyTorch, and 10.80 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)